# Laboratorio di Machine Learning per l’Analisi Finanziaria — Università Federico II
## Lezione 01 — Il churn come problema di classificazione: formulazione e analisi esplorativa (LIVE CODING)

**Autori:** Enrico Huber, Pietro Soglia  \\n
**Email:** enricohuber5@gmail.com, pietro.soglia@gmail.com  \\n
**Ultimo aggiornamento:** 2026-03-02

Nota: questa versione contiene **gap intenzionali** per il live coding.

## 1. Obiettivi di apprendimento

Al termine della lezione, dovresti saper:
- tradurre il churn in un problema con **unità statistica**, **feature** e **target**
- riconoscere i principali tipi di variabili (numeriche / categoriche / binarie)
- condurre una EDA guidata: qualità del dato, distribuzioni, sbilanciamento, segmenti
- individuare possibili criticità (es. leakage) e formulare domande di verifica sui dati

## 2. Setup

### Strumenti (cenni)
- GenAI / Code Assistants: scaffolding, debug, refactor (sempre verificare).
- Git & GitHub: versionamento e collaborazione (cenni).

### Contratto di input/output (artifact-based)
Questo notebook legge:
- `data/archive.zip` (contiene `Customer-Churn-Records.csv`)

Questo notebook scrive (sezione 3–4):
- `outputs/data/lesson_01_raw.parquet` (se disponibile) **oppure** `outputs/data/lesson_01_raw.csv` (fallback)
- `outputs/figures/lesson_01_*.png` (figure EDA con prefisso `lesson_01_`)
- `outputs/config/lesson_01_eda_summary.json` (riassunto EDA)

In [1]:
from __future__ import annotations

from pathlib import Path
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Convenzioni di progetto
TARGET_COL = 'Exited'

plt.rcParams.update({
    'figure.figsize': (8, 4),
    'axes.grid': True,
})

def find_project_root(start: Path) -> Path:
    """Cerca la root del repo risalendo finché trova README.md e requirements.txt."""
    current = start.resolve()
    for parent in [current, *current.parents]:
        if (parent / 'README.md').exists() and (parent / 'requirements.txt').exists():
            return parent
    # fallback: cwd
    return current

PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data'
ARCHIVE_PATH = DATA_DIR / 'archive.zip'

OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
OUT_DATA_DIR = OUTPUTS_DIR / 'data'
OUT_CONFIG_DIR = OUTPUTS_DIR / 'config'
OUT_FIGURES_DIR = OUTPUTS_DIR / 'figures'

for d in [OUT_DATA_DIR, OUT_CONFIG_DIR, OUT_FIGURES_DIR]:
TARGET_DIST_FIG_PATH = OUT_FIGURES_DIR / 'lesson_01_target_distribution.png'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('ARCHIVE_PATH:', ARCHIVE_PATH)
print('RAW_PARQUET_PATH:', RAW_PARQUET_PATH)

IndentationError: expected an indented block after 'for' statement on line 36 (4251029191.py, line 37)

## 3. Data loading

(Live coding) Carichiamo il CSV dall’archivio zip e ispezioniamo schema e missingness.

In [ ]:
if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(f"Non trovo {ARCHIVE_PATH}. Assicurati di avere `data/archive.zip` nel repository.")

CSV_MEMBER = 'Customer-Churn-Records.csv'

def load_csv_from_zip(archive_path: Path, member_name: str) -> pd.DataFrame:
    with zipfile.ZipFile(archive_path, 'r') as zf:
        if member_name not in zf.namelist():
            raise FileNotFoundError(
                f"Il file '{member_name}' non è presente in {archive_path.name}. Contenuti: {zf.namelist()}"
            )
        with zf.open(member_name) as f:
            return pd.read_csv(f)

df_raw = load_csv_from_zip(ARCHIVE_PATH, CSV_MEMBER)

print('Shape:', df_raw.shape)
display(df_raw.head())
display(df_raw.dtypes)

In [ ]:
# Missingness overview
missing = (df_raw.isna().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]

if len(missing) == 0:
    print('Nessun valore mancante (0% missing)')
else:
    display(missing.to_frame(name='missing_%'))

# Salva una copia efficiente dei dati grezzi (artifact)
try:
    df_raw.to_parquet(RAW_PARQUET_PATH, index=False)
    print('Salvato:', RAW_PARQUET_PATH)
except Exception as e:
    fallback_path = RAW_PARQUET_PATH.with_suffix('.csv')
    df_raw.to_csv(fallback_path, index=False)
    print('Parquet non disponibile (es. manca pyarrow).')
    print('Salvato fallback CSV:', fallback_path)
    print('Dettagli errore:', repr(e))

## 4. Analisi esplorativa (EDA)

(Live coding) Distribuzione del target, statistiche descrittive, feature principali e sbilanciamento.

### 4.1 Qualità del dataset (sanity checks)

(Live coding) Duplicati, colonne ID-like, cardinalità categoriche.

In [ ]:
# Shape, dtypes, head
print('Shape:', df_raw.shape)
display(df_raw.head())
display(df_raw.dtypes.to_frame('dtype'))

# Missingness
missing = (df_raw.isna().mean() * 100).sort_values(ascending=False)
if float(missing.max()) == 0.0:
    print('Nessun valore mancante (0% missing)')
else:
    display(missing.to_frame('missing_%'))

# Duplicati (righe e CustomerId se presente)
n_dup_rows = int(df_raw.duplicated().sum())
print('Duplicati di righe:', n_dup_rows)

if 'CustomerId' in df_raw.columns:
    n_dup_customer = int(df_raw['CustomerId'].duplicated().sum())
    print('Duplicati di CustomerId:', n_dup_customer)

# Cardinalità (quanti valori distinti per colonna)
n_rows, n_cols = df_raw.shape
nunique = df_raw.nunique(dropna=False).sort_values(ascending=False)
display(nunique.to_frame('n_unique'))

# Colonne costanti
constant_cols = nunique[nunique == 1].index.tolist()
if len(constant_cols) > 0:
    print('Colonne costanti:', constant_cols)
else:
    print('Nessuna colonna costante')

# Colonne “ID-like”: molte uniche e discrete (attenzione: float continui possono avere molti valori unici)
id_like_cols = []
for col, n_u in nunique.items():
    if col == TARGET_COL:
        continue
    if n_u >= 0.95 * n_rows:
        is_discrete = pd.api.types.is_integer_dtype(df_raw[col]) or pd.api.types.is_object_dtype(df_raw[col])
        if is_discrete:
            id_like_cols.append(col)
print('Colonne ID-like (>=95% unici, discrete):', id_like_cols)

# Cardinalità delle categoriche (utile per capire complessità delle categorie)
cat_cols_all = df_raw.select_dtypes(include=['object', 'string']).columns.tolist()
if len(cat_cols_all) == 0:
    print('Nessuna colonna categorica di tipo object/string')
else:
    card = nunique[cat_cols_all].sort_values(ascending=False)
    display(card.to_frame('n_unique'))

**Osservazioni (qualità dati)**

- Il dataset ha **10.000 righe** e **18 colonne**; non risultano duplicati di righe né duplicati di `CustomerId` → buona qualità di base per l’analisi.
- Le colonne **ID-like** (es. `RowNumber`, `CustomerId`) sono identificativi: utili per tracciare i record, ma non descrivono il profilo del cliente.
- `Surname` ha cardinalità alta (**2932**): è difficile da riassumere “per categoria”; vale la pena capire se contiene informazione reale o se è rumore/ID mascherato.
- `Geography` (3) e `Gender` (2) sono ottime variabili per segmentare e confrontare tassi di churn tra gruppi.

In [ ]:
# Target: nel dataset è tipicamente `Exited` (0/1)
if TARGET_COL not in df_raw.columns:
    raise KeyError(f"Colonna target '{TARGET_COL}' non trovata. Colonne: {list(df_raw.columns)}")

target_counts = df_raw[TARGET_COL].value_counts(dropna=False).sort_index()
target_perc = target_counts / len(df_raw) * 100

display(pd.DataFrame({'count': target_counts, 'perc_%': target_perc.round(2)}))

fig, ax = plt.subplots()
ax.bar(target_counts.index.astype(str), target_counts.values)
ax.set_title(f'Distribuzione del target ({TARGET_COL})')
ax.set_xlabel('Classe')
ax.set_ylabel('Conteggio')
fig.tight_layout()
fig.savefig(TARGET_DIST_FIG_PATH, dpi=150)
plt.show()
print('Salvata figura:', TARGET_DIST_FIG_PATH)

**Interpretazione (target e sbilanciamento)**

- Qui la churn (`Exited=1`) è **20,38%** (2038 su 10.000), mentre `Exited=0` è **79,62%**.
- Questo tasso “di base” è un numero chiave: quando confrontiamo segmenti (es. per Paese o per età) conviene sempre ricordare qual è la media complessiva.
- Lo sbilanciamento ~80/20 rende importante ragionare in termini di **tassi** e **conteggi** (non solo “impressioni” visive): differenze di pochi punti percentuali possono comunque coinvolgere molte persone.

In [ ]:
# Statistiche descrittive (numeriche)
display(df_raw.describe(include='number').T)

# Statistiche per classe (media per feature numeriche)
num_cols = df_raw.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c != TARGET_COL]

group_means = df_raw.groupby(TARGET_COL)[num_cols].mean(numeric_only=True)
display(group_means.T.sort_values(by=1, ascending=False).head(15))

**Spunti dai confronti per classe (medie numeriche)**

Le medie per classe (tabella sopra) suggeriscono alcune differenze interessanti (da verificare con grafici/segmentazioni):

- `Age`: in media i churner sono **più anziani** (circa **44,84** vs **37,41**).
- `Balance`: in media è **più alto** per `Exited=1` (circa **91.109** vs **72.743**).
- `IsActiveMember`: la quota di attivi è **più bassa** tra i churner (circa **0,361** vs **0,555**).
- `Complain`: differenza enorme (**~0,998** per `Exited=1` vs **~0,001** per `Exited=0`). Questo è un fortissimo indizio di **leakage**: la variabile potrebbe essere osservabile *dopo* (o in concomitanza di) una decisione di churn.

Nota: `CustomerId` e `RowNumber` compaiono perché sono numeriche, ma sono **identificativi**: è meglio non interpretarli come “driver” del churn.

In [ ]:
# Esempio: distribuzione età per classe
if 'Age' in df_raw.columns:
    fig, ax = plt.subplots()
    for cls, color in [(0, 'tab:blue'), (1, 'tab:orange')]:
        subset = df_raw.loc[df_raw[TARGET_COL] == cls, 'Age']
        ax.hist(subset, bins=20, alpha=0.6, label=f'{TARGET_COL}={cls}', color=color)
    ax.set_title('Distribuzione Age per classe')
    ax.set_xlabel('Age')
    ax.set_ylabel('Frequenza')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('Colonna Age non presente: salto il grafico.')

**Interpretazione (Age)**

- La distribuzione di `Age` tende a essere “spostata a destra” per `Exited=1`: coerente con la differenza nelle medie osservata sopra.
- Anche se la differenza è evidente, le due distribuzioni **si sovrappongono molto** → non esiste una soglia perfetta su `Age` da sola.
- Spunto: guardare il churn rate per **fasce** di età (quantili) per capire se l’effetto è monotono o se ci sono segmenti (es. “over 50”).

### 4.2 Distribuzioni numeriche per classe

(Live coding) Aggiungiamo altre feature numeriche (Balance, CreditScore, EstimatedSalary), boxplot e segmentazione per quantili.

In [ ]:
def plot_hist_by_target(df: pd.DataFrame, col: str, target_col: str, bins: int, out_path: Path) -> None:
    """Istogramma sovrapposto per una feature numerica, separando per classe target."""
    fig, ax = plt.subplots()
    for cls, color in [(0, 'tab:blue'), (1, 'tab:orange')]:
        subset = df.loc[df[target_col] == cls, col].dropna()
        ax.hist(subset, bins=bins, alpha=0.6, label=f'{target_col}={cls}', color=color)
    ax.set_title(f'Distribuzione {col} per classe')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequenza')
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    print('Salvata figura:', out_path)

NUM_PLOT_COLS = [c for c in ['Balance', 'CreditScore', 'EstimatedSalary'] if c in df_raw.columns]
print('Colonne numeriche plottate:', NUM_PLOT_COLS)

In [ ]:
# Plot 1: Balance
if 'Balance' in df_raw.columns:
    plot_hist_by_target(
        df_raw,
        col='Balance',
        target_col=TARGET_COL,
        bins=30,
        out_path=OUT_FIGURES_DIR / 'lesson_01_balance_hist_by_class.png',
    )

In [ ]:
# Plot 2: CreditScore
if 'CreditScore' in df_raw.columns:
    plot_hist_by_target(
        df_raw,
        col='CreditScore',
        target_col=TARGET_COL,
        bins=30,
        out_path=OUT_FIGURES_DIR / 'lesson_01_credit_score_hist_by_class.png',
    )

In [ ]:
# Plot 3: EstimatedSalary
if 'EstimatedSalary' in df_raw.columns:
    plot_hist_by_target(
        df_raw,
        col='EstimatedSalary',
        target_col=TARGET_COL,
        bins=30,
        out_path=OUT_FIGURES_DIR / 'lesson_01_estimated_salary_hist_by_class.png',
    )

**Interpretazione (feature numeriche: Balance, CreditScore, EstimatedSalary)**

- `Balance`: è una delle feature che separa meglio le classi in media (circa **91k** vs **73k**). Nei grafici spesso si nota anche un “muro” a `Balance=0`: può essere utile trattare `Balance==0` come un **segmento** oltre al valore continuo.
- `CreditScore`: la differenza tra classi è più piccola (medie ~**645** vs ~**652**) → effetto debole, ma potrebbe diventare più chiaro se guardiamo segmenti o relazioni con altre variabili.
- `EstimatedSalary`: medie molto vicine (~**101,5k** vs ~**99,7k**) → da sola non sembra un forte segnale.

Messaggio EDA: quando due distribuzioni si sovrappongono molto, è più informativo combinare più viste (istogrammi, boxplot, segmenti per fasce) invece di cercare una “regola” su una singola variabile.

In [ ]:
# Boxplot (esempio): Age e Balance per classe
cols_for_box = [c for c in ['Age', 'Balance'] if c in df_raw.columns]
if len(cols_for_box) > 0:
    fig, axes = plt.subplots(1, len(cols_for_box), figsize=(5 * len(cols_for_box), 4), sharey=False)
    if len(cols_for_box) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols_for_box):
        data0 = df_raw.loc[df_raw[TARGET_COL] == 0, col].dropna()
        data1 = df_raw.loc[df_raw[TARGET_COL] == 1, col].dropna()
        ax.boxplot([data0, data1], labels=[f'{TARGET_COL}=0', f'{TARGET_COL}=1'])
        ax.set_title(f'Boxplot {col} per classe')
        ax.set_xlabel('Classe')
        ax.set_ylabel(col)
    fig.tight_layout()
    out_path = OUT_FIGURES_DIR / 'lesson_01_boxplots_age_balance.png'
    fig.savefig(out_path, dpi=150)
    plt.show()
    print('Salvata figura:', out_path)

**Interpretazione (boxplot)**

- Il boxplot evidenzia mediana, dispersione e outlier; è utile per confronti robusti tra classi.
- Dal run completo: `Age` e `Balance` tendono ad avere valori centrali più alti per `Exited=1`.

In [ ]:
# Segmentazione semplice: churn rate per fasce (quantili)
def churn_rate_by_quantile(df: pd.DataFrame, col: str, target_col: str, q: int = 10) -> pd.DataFrame:
    x = df[col]
    # qcut può fallire se ci sono molti valori ripetuti: gestiamo con duplicates='drop'
    bins = pd.qcut(x, q=q, duplicates='drop')
    out = (
        df.assign(_bin=bins)
          .groupby('_bin', observed=True)[target_col]
          .agg(['mean', 'count'])
          .rename(columns={'mean': 'churn_rate', 'count': 'n'})
    )
    return out.reset_index().rename(columns={'_bin': f'{col}_bin'})

cols_for_bins = [c for c in ['Age', 'Balance', 'CreditScore'] if c in df_raw.columns]
for col in cols_for_bins:
    display(churn_rate_by_quantile(df_raw, col=col, target_col=TARGET_COL, q=10).head())

**Interpretazione (segmentazione per quantili)**

Questa tabella è molto utile perché trasforma una feature continua in una lettura “business-friendly”: *quanto churn c’è in ciascuna fascia?*

- `Age`: già nelle prime fasce vediamo churn rate che cresce (es. ~**7–12%** nelle fasce più giovani mostrate), suggerendo un effetto **non costante** lungo l’età.
- `Balance`: la fascia più bassa (che include molti `Balance` vicini a 0) ha churn ~**14,75%**, mentre fasce successive arrivano a ~**25–28%** → segnale più forte e potenzialmente **monotono** (almeno in parte).
- `CreditScore`: i primi quantili mostrano churn abbastanza **piatto** (~**20–23%**), coerente con una correlazione debole.

Spunto EDA: queste tabelle aiutano anche a scegliere *come raccontare* i risultati (per fasce) e a individuare segmenti su cui concentrarsi per ulteriori approfondimenti.

In [ ]:
# Feature categoriche: distribuzione per classe (esempi)
cat_candidates = ['Geography', 'Gender', 'Card Type']
cat_cols = [c for c in cat_candidates if c in df_raw.columns]

for col in cat_cols:
    ct = pd.crosstab(df_raw[col], df_raw[TARGET_COL], normalize='index')
    display(ct.style.format('{:.2%}').set_caption(f'P({TARGET_COL}=1 | {col}) per categoria'))

**Interpretazione (categorie: $P(Exited=1 \mid categoria)$)**

- Queste tabelle mettono in evidenza differenze tra gruppi in modo immediato.
- Controllare sempre anche la numerosità: categorie rare possono essere instabili.
- Nel run completo vedremo differenze marcate in `Geography` e `Gender`.

### 4.3 Tasso di churn per categorie (e variabili binarie)

(Live coding) Usiamo barplot del churn rate per Geography/Gender e binarie come IsActiveMember/Complain.

In [ ]:
def churn_rate_by_category(df: pd.DataFrame, col: str, target_col: str) -> pd.DataFrame:
    out = (
        df.groupby(col, dropna=False)[target_col]
          .agg(['mean', 'count'])
          .rename(columns={'mean': 'churn_rate', 'count': 'n'})
          .sort_values(['churn_rate', 'n'], ascending=[False, False])
    )
    return out.reset_index()

def plot_churn_rate_bar(df_rate: pd.DataFrame, col: str, out_path: Path, top_n: int = 15) -> None:
    df_plot = df_rate.head(top_n).copy()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(df_plot[col].astype(str), df_plot['churn_rate'])
    ax.set_title(f'Churn rate per {col} (top {top_n})')
    ax.set_xlabel(col)
    ax.set_ylabel('churn rate')
    ax.set_ylim(0, min(1.0, max(0.3, float(df_plot['churn_rate'].max()) * 1.2)))
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    print('Salvata figura:', out_path)

CAT_PLOT_COLS = [c for c in ['Geography', 'Gender', 'Card Type', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'Complain'] if c in df_raw.columns]
print('Colonne categoriche/binarie analizzate:', CAT_PLOT_COLS)

In [ ]:
# Plot: churn rate per Geography
if 'Geography' in df_raw.columns:
    df_geo = churn_rate_by_category(df_raw, col='Geography', target_col=TARGET_COL)
    display(df_geo)
    plot_churn_rate_bar(df_geo, col='Geography', out_path=OUT_FIGURES_DIR / 'lesson_01_churn_rate_by_geography.png', top_n=20)

**Interpretazione (Geography, dal run completo)**

- `Germany` ~**32,44%** vs `Spain` ~**16,67%** e `France` ~**16,17%**.
- Ottimo esempio di segmentazione e di possibili fattori latenti (contesto diverso).

In [ ]:
# Plot: churn rate per Gender
if 'Gender' in df_raw.columns:
    df_gender = churn_rate_by_category(df_raw, col='Gender', target_col=TARGET_COL)
    display(df_gender)
    plot_churn_rate_bar(df_gender, col='Gender', out_path=OUT_FIGURES_DIR / 'lesson_01_churn_rate_by_gender.png', top_n=10)

**Interpretazione (Gender, dal run completo)**

- `Female` ~**25,07%** vs `Male` ~**16,47%**.
- Spunto: verificare se l’effetto resta dopo aver condizionato su `Geography`/`Age` (interazioni).

In [ ]:
# Plot: churn rate per IsActiveMember (variabile binaria)
if 'IsActiveMember' in df_raw.columns:
    df_rate = churn_rate_by_category(df_raw, col='IsActiveMember', target_col=TARGET_COL)
    display(df_rate)
    plot_churn_rate_bar(
        df_rate,
        col='IsActiveMember',
        out_path=OUT_FIGURES_DIR / 'lesson_01_churn_rate_by_isactivemember.png',
        top_n=10,
    )

**Interpretazione (IsActiveMember, dal run completo)**

- `IsActiveMember=0`: churn ~**26,87%**; `IsActiveMember=1`: churn ~**14,27%**.
- Lettura naturale: l’attività/engagement è associata a minor churn; esplorare interazioni con altri segmenti.

In [ ]:
# Plot: churn rate per Complain (variabile binaria)
if 'Complain' in df_raw.columns:
    df_rate = churn_rate_by_category(df_raw, col='Complain', target_col=TARGET_COL)
    display(df_rate)
    plot_churn_rate_bar(
        df_rate,
        col='Complain',
        out_path=OUT_FIGURES_DIR / 'lesson_01_churn_rate_by_complain.png',
        top_n=10,
    )

**Interpretazione critica (Complain = possibile leakage)**

Qui vediamo un pattern quasi “troppo bello per essere vero”:

- `Complain=1`: churn **~99,51%** (n=2044)
- `Complain=0`: churn **~0,05%** (n=7956)

Questo suggerisce che `Complain` potrebbe essere una variabile **post-evento** (es. un reclamo registrato quando il cliente è già in fase di uscita) o comunque molto vicina temporalmente alla churn.

In EDA, le azioni sensate sono:

- verificare la **definizione** della variabile e la timeline (quando è osservabile?)
- controllare se ci sono incongruenze (es. valori impossibili, cambi di definizione nel tempo, missing anomali)
- decidere se trattarla come “segnale operativo tardivo” (utile per analisi) oppure come informazione non utilizzabile in scenari di previsione anticipata.

### 4.4 Correlazioni e relazioni bivariate (solo esplorative)

(Live coding) Correlazioni con target, heatmap (top feature), scatter Age vs Balance.

In [ ]:
# Correlazione (numeriche) con il target
num_cols_all = df_raw.select_dtypes(include='number').columns.tolist()
if TARGET_COL in num_cols_all:
    num_cols_all = [c for c in num_cols_all if c != TARGET_COL]

# Rimuoviamo colonne ID-like dalla matrice numerica, se presenti (spesso non informative)
id_like_to_drop = [c for c in ['RowNumber', 'CustomerId'] if c in df_raw.columns]
num_cols_for_corr = [c for c in num_cols_all if c not in id_like_to_drop]

corr_with_target = df_raw[num_cols_for_corr + [TARGET_COL]].corr(numeric_only=True)[TARGET_COL].drop(TARGET_COL)
corr_table = corr_with_target.reindex(corr_with_target.abs().sort_values(ascending=False).index)
display(corr_table.to_frame('corr_with_target'))

# Heatmap delle correlazioni tra numeriche (top 12 per |corr con target|)
top_corr_cols = corr_table.abs().head(12).index.tolist()
corr_mat = df_raw[top_corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_mat.values, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(top_corr_cols)))
ax.set_yticks(range(len(top_corr_cols)))
ax.set_xticklabels(top_corr_cols, rotation=45, ha='right')
ax.set_yticklabels(top_corr_cols)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Correlazioni (feature numeriche selezionate)')
fig.tight_layout()
out_path = OUT_FIGURES_DIR / 'lesson_01_correlation_heatmap.png'
fig.savefig(out_path, dpi=150)
plt.show()
print('Salvata figura:', out_path)

**Interpretazione (correlazioni, dal run completo)**

- `Complain` è estremamente correlata con `Exited` (≈ 0,996): molto probabilmente leakage.
- Tra le feature “plausibili” emergono `Age` (≈ 0,285), `IsActiveMember` (≈ -0,156), `Balance` (≈ 0,119).
- Pearson misura soprattutto relazioni lineari: correlazione bassa non implica “inutile”.

In [ ]:
# Scatter bivariato (esempio): Age vs Balance, colorato per classe
if 'Age' in df_raw.columns and 'Balance' in df_raw.columns:
    fig, ax = plt.subplots(figsize=(7, 5))
    for cls, color in [(0, 'tab:blue'), (1, 'tab:orange')]:
        sub = df_raw.loc[df_raw[TARGET_COL] == cls, ['Age', 'Balance']].dropna()
        ax.scatter(sub['Age'], sub['Balance'], s=10, alpha=0.25, c=color, label=f'{TARGET_COL}={cls}')
    ax.set_title('Age vs Balance (color: classe)')
    ax.set_xlabel('Age')
    ax.set_ylabel('Balance')
    ax.legend()
    fig.tight_layout()
    out_path = OUT_FIGURES_DIR / 'lesson_01_age_vs_balance_scatter.png'
    fig.savefig(out_path, dpi=150)
    plt.show()
    print('Salvata figura:', out_path)

**Interpretazione (scatter Age vs Balance)**

- Questo grafico aiuta a vedere *insieme* due feature e capire se le differenze tra classi si concentrano in certe regioni del piano.
- Si nota una linea di punti a `Balance=0`: è una struttura particolare del dataset. In EDA è utile trattare `Balance==0` come un **segmento** a parte, oltre a guardare `Balance` come continuo.
- La sovrapposizione tra classi resta ampia: è un segnale che vale la pena integrare più viste (grafici + segmenti) prima di trarre conclusioni.

In [ ]:
# Salviamo un piccolo riassunto EDA come artifact
import json

eda_summary = {
    'n_rows': int(n_rows),
    'n_cols': int(n_cols),
    'target_col': TARGET_COL,
    'churn_rate': float(df_raw[TARGET_COL].mean()),
    'id_like_cols': id_like_cols,
}

# Aggiungiamo qualche tabella “top” se presente Geography
if 'Geography' in df_raw.columns:
    geo_rate = churn_rate_by_category(df_raw, col='Geography', target_col=TARGET_COL)
    eda_summary['churn_rate_by_geography_top'] = geo_rate.head(10).to_dict(orient='records')

# Correlazioni con target (top 10)
eda_summary['corr_with_target_top'] = (
    corr_table.abs().head(10).to_frame('abs_corr')
    .join(corr_table.to_frame('corr'), how='left')
    .reset_index()
    .rename(columns={'index': 'feature'})
    .to_dict(orient='records')
)

summary_path = OUT_CONFIG_DIR / 'lesson_01_eda_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(eda_summary, f, ensure_ascii=False, indent=2)
print('Salvato riassunto EDA:', summary_path)

**Artifact (riassunto EDA su disco)**

Abbiamo salvato un piccolo riassunto in `outputs/config/lesson_01_eda_summary.json`. È utile perché:

- rende l’EDA **riproducibile** senza dipendere dallo stato del kernel
- permette di controllare rapidamente numeri chiave (churn rate, segmenti top, correlazioni)
- è un esempio concreto di comunicazione via artifact: possiamo ripartire da file su disco anche con kernel “pulito”.

**Mini-sintesi EDA (messaggi chiave)**

- Churn rate complessivo: **20,38%** (sbilanciamento ~80/20).
- Segmenti forti: `Geography` (Germany **32,44%**), `Gender` (Female **25,07%**), `IsActiveMember` (0 **26,87%** vs 1 **14,27%**).
- Feature numeriche con segnale: `Age` e `Balance` (effetti visibili ma con sovrapposizione).
- Variabile sospetta: `Complain` (quasi deterministica) → verificare timeline e definizione per rischio leakage.
- Colonne da trattare con cura: `RowNumber`/`CustomerId` (ID-like), `Surname` (alta cardinalità).

### Discussione

Domande guida (da discutere in aula):
- Il target è sbilanciato (~80/20): come cambia il modo in cui comunichereste i risultati (tassi, conteggi, confronto tra segmenti)?
- Quali feature sembrano più diverse tra `Exited=0` e `Exited=1` (es. `Age`, `Balance`, `IsActiveMember`)? In che direzione e di quanto?
- `Complain` sembra quasi deterministica: quali indizi vi fanno pensare a leakage? Che controllo “di processo” fareste sui dati?
- Il churn rate cambia molto per `Geography` (Germany > Spain/France): quali ipotesi plausibili possono spiegare la differenza?
- `Surname` ha cardinalità molto alta: quali rischi comporta in un’analisi per categorie? Che verifiche fareste per capire se porta informazione o solo rumore?
- Se doveste proporre 2–3 azioni di retention, su quali segmenti le concentrereste e perché?

## 5. Esercizi (EDA guidata)

1. [easy] Calcola la percentuale di churn per `Geography` e ordina le categorie. (Suggerimento: usa `groupby` + `mean` sul target.)
2. [easy] Visualizza un boxplot di `Balance` per classe `Exited`.
3. [medium] Costruisci una tabella di churn rate per decili di `Age` (usa `pd.qcut`) e commenta quali fasce sembrano più a rischio.
4. [medium] Scegli 3 feature e discuti (in 5 righe) perché potrebbero essere associate al churn *alla luce di grafici e tabelle EDA* (senza parlare di causalità).
5. [medium] Cerca colonne potenzialmente ‘pericolose’ (leakage) e motiva la scelta. Quali segnali cercheresti?

## 6. Riepilogo

- Il churn è descritto dal target binario `Exited` e nel dataset vale **20,38%**.
- L’EDA serve a capire qualità del dato (missingness, duplicati, ID-like), distribuzioni e differenze tra classi.
- Segmenti con tassi diversi (`Geography`, `Gender`, `IsActiveMember`) suggeriscono dove guardare più a fondo e come impostare una discussione “business”.
- Alcune colonne possono nascondere leakage (es. `Complain`): è cruciale controllare definizioni e timeline.
- Gli artifact salvati in `outputs/` rendono l’analisi ripetibile e indipendente dallo stato del kernel.